# Environmental Sound Recognition — Training

**QUAN TRỌNG**: Trước khi chạy, hãy đảm bảo:
1. **Settings → Accelerator → GPU T4 x2** (hoặc P100)
2. **Settings → Internet → ON** (để download pretrained weights)

In [ ]:
# Cell 1: Kiểm tra GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    mem = torch.cuda.get_device_properties(i).total_mem / 1e9
    print(f"         Memory: {mem:.1f} GB")

# Test thực tế: chạy 1 phép tính trên GPU
if torch.cuda.is_available():
    x = torch.randn(1000, 1000, device='cuda')
    y = x @ x
    print(f"\n✅ GPU test PASSED — tensor trên {y.device}")
else:
    print("\n❌ GPU KHÔNG khả dụng! Vào Settings → Accelerator → chọn GPU")

In [ ]:
# Cell 2: Clone repo
!git clone https://github.com/TotallyNotMinh/Environmental-Sound-Recognition-Under-Unseen-Conditions.git
%cd /kaggle/working/Environmental-Sound-Recognition-Under-Unseen-Conditions
!git fetch origin
!git pull origin main

In [ ]:
# Cell 3: Install dependencies
!pip install -q h5py scipy opencv-python tqdm torchinfo

In [ ]:
# Cell 4: Cấu hình NCCL cho Kaggle (fix distributed training)
import os
os.environ['NCCL_P2P_DISABLE'] = '1'       # Tắt P2P (Kaggle container không hỗ trợ)
os.environ['NCCL_IB_DISABLE'] = '1'        # Tắt InfiniBand
os.environ['NCCL_SOCKET_IFNAME'] = 'lo'    # Dùng loopback interface
os.environ['NCCL_DEBUG'] = 'WARN'          # Log NCCL warnings
print("NCCL environment configured for Kaggle.")

In [ ]:
# Cell 5: Training — tự detect số GPU

import torch
NUM_GPUS = torch.cuda.device_count()
print(f"Detected {NUM_GPUS} GPU(s)")

if NUM_GPUS >= 2:
    # Multi-GPU (T4 x2)
    print("→ Chạy Distributed Training trên 2 GPUs")
    !NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 NCCL_SOCKET_IFNAME=lo \
        torchrun --nproc_per_node=2 scripts/train_classifier.py \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --batch-size 24 \
        --checkpoint-dir /kaggle/working/checkpoints/ \
        --encoder-lr 2e-4 \
        --head-lr 2e-4
elif NUM_GPUS == 1:
    # Single GPU
    print("→ Chạy Single-GPU Training")
    !python scripts/train_classifier.py \
        --data-path /kaggle/input/datasets/yousirui1/fsd50k/fsd50k \
        --batch-size 48 \
        --checkpoint-dir /kaggle/working/checkpoints/ \
        --encoder-lr 2e-4 \
        --head-lr 2e-4
else:
    print("❌ Không có GPU! Vào Settings → Accelerator → chọn GPU")